# OGBN-ArXiv Feature Engineering for Traditional ML

This notebook performs feature engineering on the OGBN-ArXiv dataset to preserve graph structural information in tabular format. The goal is to extract graph topology features that can be used with traditional ML algorithms like XGBoost and standard neural networks, avoiding the need for specialized graph neural networks.

In [2]:
from ogb.nodeproppred import NodePropPredDataset
import pandas as pd
import networkx as nx
import torch

## Load the OGBN-ArXiv Dataset
The dataset contains:
- **Nodes**: ~169K research papers
- **Edges**: ~1.2M citation links
- **Node Features**: 128-dimensional embeddings derived from paper abstracts
- **Labels**: 40 subject area classes

In [3]:
_original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

# Apply patch
torch.load = patched_torch_load

# Load dataset
dataset = NodePropPredDataset(name="ogbn-arxiv", root="../../data/ogbn")
graph, labels = dataset[0]

# Restore original (optional)
torch.load = _original_torch_load

labels

array([[ 4],
       [ 5],
       [28],
       ...,
       [10],
       [ 4],
       [ 1]])

## Extract Graph Topology as Tabular Features

We convert the edge list to a NetworkX graph and compute structural features that capture the graph topology in tabular format. This allows us to preserve critical graph information for use with traditional ML algorithms like XGBoost and standard neural networks.

### Graph Features Extracted:
1. **Degree Centrality**: Number of connections (citations and cited by) for each paper
   - Captures local connectivity and paper influence
   - Essential for XGBoost to understand node importance

2. **PageRank**: Global importance based on the entire citation network
   - Provides a recursive measure of paper significance
   - Helps traditional ML models understand network-wide influence patterns

3. **Clustering Coefficient**: Local network density around each node
   - Captures community structure information
   - Indicates whether papers are in dense research clusters

4. **Betweenness Centrality**: Measures a bridging role between different parts of the network. Not included as it takes 1.5+ hours to complete.
   - Identifies papers that connect different research areas
   - Critical for understanding interdisciplinary connections

In [11]:
edge_index = graph["edge_index"]
node_feat = graph["node_feat"]

In [26]:
G = nx.Graph()
G.add_edges_from(edge_index.T.tolist())

features = {
    'degree': dict(G.degree()),
    'pagerank': nx.pagerank(G),
    'clustering': nx.clustering(G),
}

df_graph = pd.DataFrame(features)

In [34]:
correlation_data = df_graph.merge(labels, left_index=True, right_index=True)
corr = correlation_data.corr()['label'].drop('label')
print(corr)

degree        0.007333
pagerank      0.006602
clustering    0.035005
Name: label, dtype: float64


## Create ML-Ready Dataset

We combine the original node features with our extracted graph topology features to create a comprehensive tabular dataset:

1. **Content Features**: 128-dimensional embeddings from paper abstracts
2. **Graph Topology Features**: Structural properties that capture the citation network
3. **Labels**: Ground truth subject area classifications

In [31]:
df_feats = pd.DataFrame(node_feat)

data = df_feats.merge(df_graph, left_index=True, right_index=True)
labels = pd.DataFrame(data=labels, columns=['label'])

data.head()

,0,1,2,3,4,5,6,7,8,9,...,121,122,123,124,125,126,127,degree,pagerank,clustering
0,-0.057943,-0.052530,-0.072603,-0.026555,0.130435,-0.241386,-0.449242,-0.018443,-0.087218,0.112320,...,0.053230,0.332873,0.104175,0.007408,0.173364,-0.172796,-0.140059,291,0.000062,0.034364
1,-0.124500,-0.070665,-0.325202,0.007779,-0.001559,0.074189,-0.191013,0.049689,0.026369,0.099364,...,0.021567,0.281503,-0.173423,0.202082,0.068524,-0.372111,-0.301036,2,0.000003,1.000000
2,-0.080242,-0.023328,-0.183787,-0.180707,0.075765,-0.125818,-0.394573,-0.219078,-0.108931,0.056966,...,-0.214012,0.182186,-0.121589,-0.073642,0.109919,0.117589,-0.139883,14,0.000007,0.208791
3,-0.145044,0.054915,-0.126666,0.039971,-0.055909,-0.101278,-0.339202,-0.115801,-0.080058,-0.001633,...,-0.226921,0.188418,-0.017295,0.063449,0.017816,0.085364,-0.081804,2,0.000003,1.000000
4,-0.071154,0.070766,-0.281432,-0.161892,-0.165246,-0.029116,-0.338593,-0.138727,0.100015,0.132794,...,0.026462,0.376349,-0.253772,0.084472,0.098033,-0.075347,-0.111687,6,0.000004,0.800000


## Inspect the Final Feature Set

Let's examine the structure of our final feature set to ensure everything looks correct.

In [19]:
data.keys()

Index([           0,            1,            2,            3,            4,
                  5,            6,            7,            8,            9,
       ...
                122,          123,          124,          125,          126,
                127,    'node_id',     'degree',   'pagerank', 'clustering'],
      dtype='object', length=132)

## Save the ML-Ready Dataset

In [35]:
df_feats.to_csv('../../data/ogbn/processed/data.csv', index=False)
labels.to_csv('../../data/ogbn/processed/labels.csv', index=False)

# Conclusion
The correlations between graph-structural features and paper labels are close to zero:
- Degree: 0.007
- PageRank: 0.007
- Clustering: 0.035

This indicates that citation connectivity patterns are relatively uniform across research areas.
In other words, papers from different fields show similar structural behavior in the graph,
so these features do not provide strong predictive signals for classification.
